In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['image.aspect'] = 'equal'
plt.rcParams['image.cmap'] = 'gray'
plt.rcParams['figure.dpi'] = 100
import warnings
warnings.filterwarnings('ignore')

# Réhaussement et visualisation d'images

Assurez-vous de lire ce préambule avant d'exécutez le reste du notebook.

## Préambule

### Objectifs

Dans ce chapitre, nous abordons quelques techniques de réhaussement et de visualisation d'images. Ce chapitre est aussi disponible sous la forme d'un notebook Python:

[![](images/colab.png)](https://colab.research.google.com/github/sfoucher/TraitementImagesPythonVol1/blob/main/notebooks/02-RehaussementVisualisationImages.ipynb)


### 

### Bibliothèques

Les bibliothèques qui vont être explorées dans ce chapitre sont les suivantes:

-   [SciPy](https://scipy.org/)

-   [NumPy](https://numpy.org/)

-   [opencv-python · PyPI](https://pypi.org/project/opencv-python/)

-   [scikit-image](https://scikit-image.org/)

-   [Rasterio](https://rasterio.readthedocs.io/en/stable/)

-   [Xarray](https://docs.xarray.dev/en/stable/)

-   [rioxarray](https://corteva.github.io/rioxarray/stable/index.html)

Dans l'environnement Google Colab, seul `rioxarray` et GDAL doivent être installés:

In [ ]:
%%capture --no-stderr
!apt-get update
!apt-get install gdal-bin libgdal-dev

Dans l'environnement [Google Colab](https://colab.research.google.com/), il convient de s'assurer que les librairies sont installées:

In [ ]:
%%capture --no-stderr
!pip install -qU matplotlib rioxarray xrscipy scikit-image leafmap localtileserver

Vérifier les importations:

In [ ]:
import numpy as np
import rioxarray as rxr
from scipy import signal
import xarray as xr
import xrscipy
import matplotlib.pyplot as plt

### Données

Nous utiliserons les images suivantes dans ce chapitre:

In [ ]:
%%capture --no-stderr
import gdown

gdown.download('https://drive.google.com/uc?export=download&confirm=pbef&id=1a6Ypg0g1Oy4AJt9XWKWfnR12NW1XhNg_', output= 'RGBNIR_of_S2A.tif')
gdown.download('https://drive.google.com/uc?export=download&confirm=pbef&id=1a6O3L_abOfU7h94K22At8qtBuLMGErwo', output= 'sentinel2.tif')
gdown.download('https://drive.google.com/uc?export=download&confirm=pbef&id=1_zwCLN-x7XJcNHJCH6Z8upEdUXtVtvs1', output= 'berkeley.jpg')
gdown.download('https://drive.google.com/uc?export=download&confirm=pbef&id=1dM6IVqjba6GHwTLmI7CpX8GP2z5txUq6', output= 'SAR.tif')
gdown.download('https://drive.google.com/uc?export=download&confirm=pbef&id=1a4PQ68Ru8zBphbQ22j0sgJ4D2quw-Wo6', output= 'landsat7.tif')

Vérifiez que vous êtes capable de les lire :

In [ ]:

with rxr.open_rasterio('berkeley.jpg', mask_and_scale= True) as img_rgb:
    print(img_rgb)
with rxr.open_rasterio('sentinel2.tif', mask_and_scale= True) as img_s2:
    print(img_s2)
with rxr.open_rasterio('RGBNIR_of_S2A.tif', mask_and_scale= True) as img_rgbnir:
    print(img_rgbnir)
with rxr.open_rasterio('SAR.tif', mask_and_scale= True) as img_SAR:
    print(img_SAR)

## Visualisation en Python

D'emblée, il faut mentionner que Python n'est pas vraiment fait pour visualiser de la donnée de grande taille, le niveau d'interactivité est aussi assez limité. Pour une visualisation interactives, il est plutôt conseillé d'utiliser un outil comme [QGIS](https://qgis.org/). Néanmoins, il est possible de visualiser de petites images avec la librairie [`matplotlib`](https://matplotlib.org/stable/) qui est la librairie principale de visualisation en Python. Cette librairie est extrêmement riche et versatile, nous ne présenterons que les bases nécessaires pour démarrer. Le lecteur désirant aller plus loin pourra consulter les nombreux tutoriels disponibles comme [celui-ci](https://matplotlib.org/stable/tutorials/index.html).

La fonction de base pour créer une figure est `subplots`, la largeur et la hauteur en pouces de la figure peuvent être contrôlées via le paramètre `figsize`:

In [ ]:
import matplotlib.pyplot as plt
fig, ax= plt.subplots(figsize=(5, 4))
plt.show()

Pour l'affichage des images, la fonction `imshow` permet d'afficher une matrice 2D à une dimension en format *float* ou une matrice RVB avec 3 bandes. Il est important que les dimensions de la matrice soient dans l'ordre hauteur, largeur et bande.

In [ ]:
import matplotlib.pyplot as plt
fig, ax= plt.subplots(figsize=(6, 5))
plt.imshow(img_rgbnir[0].data)
plt.show()

Pour un affichage à trois bandes, les valeurs seront ramenées sur une dynamique de 0 à 1, il est donc nécessaire de normaliser les valeurs avant l'affichage:

In [ ]:
import matplotlib.pyplot as plt
fig, ax= plt.subplots(figsize=(6, 5))
# ordre des bandes B,V,R,PIR -> on réordonne en R,V,B pour un affichage en vraies couleurs
plt.imshow(img_rgbnir.sel(band=[3,2,1]).data.transpose(1,2,0)/2500.0)
plt.show()

On remarquera les valeurs des axes `x` et `y` avec une origine en haut à gauche. Ceci est un référentiel purement matriciel (lignes et colonnes); autrement dit, il n'y a pas ici de géoréférence. Pour pallier à cette limitation, les librairies `rasterio` et `xarray` proposent une extension de la fonction `imshow` permettant d'afficher les coordonnées cartographiques ainsi qu'un contrôle la dynamique de l'image:

In [ ]:
import rioxarray as rxr
fig, ax= plt.subplots(figsize=(6, 5))
img_rgbnir.sel(band=[3,2,1]).plot.imshow(vmin=86, vmax=5000)  # R,V,B (ordre stocké B,V,R,PIR)
ax.set_title('Imshow avec rioxarray')
plt.show()

### Visualisation sur le Web

Les affichages `matplotlib` précédents sont **statiques**. Pour explorer une image de manière **interactive** — zoomer, se déplacer, superposer un fond de carte, comparer deux visualisations — on peut la placer sur une carte web. La librairie [`leafmap`](https://leafmap.org/) offre une interface Python unifiée au-dessus de `folium` et `ipyleaflet` et permet, en quelques lignes, d'afficher un GeoTIFF géoréférencé sur une carte glissante (*slippy map*).



On crée une carte, puis on ajoute directement notre image locale. Comme les bandes sont stockées dans l'ordre B, V, R, PIR, on demande les indices `[3, 2, 1]` pour un composé **vraie couleur** :

In [ ]:
import leafmap

m = leafmap.Map()
m.add_raster('RGBNIR_of_S2A.tif', indexes=[3, 2, 1], layer_name='Vraie couleur')
m

Cette carte est **interactive** : elle ne s'affiche que dans un notebook. Pour le livre, nous en produisons une **capture statique** hors ligne avec `matplotlib` et [`contextily`](https://contextily.readthedocs.io/) (qui télécharge le fond OpenStreetMap). L'image Sentinel-2 est reprojetée en Web Mercator (`EPSG:3857`), chaque bande est étirée entre ses centiles 2 % et 98 %, et les pixels *no data* (valeur `65535` après reprojection) sont rendus transparents pour laisser voir le fond de carte :

In [ ]:
import numpy as np, rioxarray as rxr, contextily as cx
import matplotlib.pyplot as plt

src = rxr.open_rasterio('RGBNIR_of_S2A.tif').rio.reproject('EPSG:3857')
nd = src.rio.nodata                          # 65535 après reprojection
x, y = src.x.values, src.y.values
extent = [x.min(), x.max(), y.min(), y.max()]

def composite(bandes):                       # composé RGBA étiré, no_data transparent
    raw = src.isel(band=[b - 1 for b in bandes]).to_numpy().astype('float32')
    rgba = np.zeros(raw.shape[1:] + (4,), 'float32')
    for i in range(3):
        canal = raw[i]; valide = canal != nd
        p2, p98 = np.percentile(canal[valide], [2, 98])
        rgba[..., i] = np.clip((canal - p2) / (p98 - p2), 0, 1)
    rgba[..., 3] = (raw != nd).all(axis=0)   # no_data -> transparent
    return rgba

fig, ax = plt.subplots(figsize=(7, 6))
cx.add_basemap(ax, crs='EPSG:3857', source=cx.providers.OpenStreetMap.Mapnik, zorder=0)
ax.imshow(composite([3, 2, 1]), extent=extent, origin='upper', zorder=1)
ax.axis('off'); plt.show()


La méthode `split_map` crée un **comparateur à volet glissant**, idéal pour opposer deux visualisations de la même scène — ici la vraie couleur (`[3, 2, 1]`) et l'infrarouge fausses couleurs (`[4, 3, 2]`), qui fait ressortir la végétation en rouge :

In [ ]:
m = leafmap.Map()
m.split_map(
    left_layer='RGBNIR_of_S2A.tif',
    right_layer='RGBNIR_of_S2A.tif',
    left_args={'indexes': [3, 2, 1]},
    right_args={'indexes': [4, 3, 2]},
    left_label='Vraie couleur', right_label='Infrarouge',
)
m

Pour le livre, la capture statique correspondante réutilise la fonction `composite` définie plus haut, appliquée aux deux composés côte à côte :

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, bandes, titre in zip(axes, ([3, 2, 1], [4, 3, 2]),
                             ('Vraie couleur', 'Infrarouge (PIR, R, V)')):
    cx.add_basemap(ax, crs='EPSG:3857', source=cx.providers.OpenStreetMap.Mapnik, zorder=0)
    ax.imshow(composite(bandes), extent=extent, origin='upper', zorder=1)
    ax.set_title(titre); ax.axis('off')
plt.tight_layout(); plt.show()


`leafmap` permet aussi d'ajouter un fond satellite (`m.add_basemap('Esri.WorldImagery')`), de charger des images distantes au format *Cloud Optimized GeoTIFF* (`m.add_cog_layer(url)`), ou d'inspecter les valeurs de pixels au clic (`m.add('inspector')`). C'est un outil précieux pour situer une image ou un résultat de traitement (comme une classification, @sec-chap05) dans son contexte géographique.



## Réhaussements visuels

Le réhaussement visuel d'une image vise principalement à améliorer la qualité visuelle d'une image en améliorant le contraste, la dynamique ou la texture d'une image. De manière générale, ce réhaussement ne modifie pas la donnée d'origine mais il est appliquée dynamiquement à l'affichage pour des fins d'inspection visuelle. Le réhaussement nécessite généralement une connaissance des caractéristiques statistiques d'une image. Ces statistiques sont ensuite exploitées pour appliquer diverses transformations linéaires ou non linéaires.

### Statistiques d'une image

On peut considérer un ensemble de statistique pour chacune des bandes d'une image:

-   valeurs minimales et maximales

-   valeurs moyennes,

-   Quartiles (1er quartile, médiane et 3ième quartile), quantiles et percentiles.

-   écart-type, et coefficients d'asymétrie (*skewness*) et d'applatissement (*kurtosis*)

Ces statistiques doivent être calculées pour chaque bande d'une image multispectrale.

En ligne de commande, `gdalinfo` permet d'interroger rapidement un fichier image pour connaitre ces statistiques univariées de base:

In [ ]:
!gdalinfo -stats landsat7.tif

Les librairies de base comme `rasterio` et `xarray` produisent facilement un sommaire des statistiques de base avec la fonction [stats](https://rasterio.readthedocs.io/en/stable/api/rasterio.io.html#rasterio.io.BufferedDatasetWriter.stats):

In [ ]:

import rasterio as rio
import numpy as np
with rio.open('landsat7.tif') as src:
    stats= src.stats()
    print(stats)

La librairie `xarray` donne accès à des fonctionnalités plus sophistiquées comme le calcul des quantiles:

In [ ]:
import rioxarray as riox
with riox.open_rasterio('landsat7.tif', masked= True) as src:
    print(src)
quantiles = src.quantile(dim=['x','y'], q=[.025,.25,.5,.75,.975])
quantiles

#### Calcul de l'histogramme

Le calcul d'un histogramme pour une image (une bande) permet d'avoir une vue plus détaillée de la répartition des valeurs radiométriques. Le calcul d'un histogramme nécessite minimalement de faire le choix du nombre de barre ( *bins* ou de la largeur ). Un *bin* est un intervalle de valeurs pour lequel on peut calculer le nombre de valeurs observées dans l'image. La fonction de base pour ce type de calcul est la fonction `numpy.histogram()`:

In [ ]:
import numpy as np
array = np.random.randint(0,10,100) # 100 valeurs aléatoires entre 0 et 10
hist, bin_limites = np.histogram(array, density=True)
print('valeurs :',hist)
print('limites :',bin_limites)

Le calcul se fait avec 10 intervalles par défaut.

In [ ]:
fig, ax= plt.subplots(figsize=(5, 4))
plt.bar(bin_limites[:-1],hist)
plt.show()

Pour des besoins de visualisation, le calcul des valeurs extrêmes de l'histogramme peut aussi se faire via les quantiles comme discutés auparavant.

##### Visualisation des histogrammes

La librarie `rasterio` est probablement l'outil le plus simples pour visualiser rapidement des histogrammes sur une image multi-spectrale:

In [ ]:
import rasterio as rio
from rasterio.plot import show_hist
with rio.open('RGBNIR_of_S2A.tif') as src:
  show_hist(src, bins=50, lw=0.0, stacked=False, alpha=0.3,histtype='stepfilled', title="Histogram")

### Réhaussements linéaires

Le réhaussement linéaire (*linear stretch*) d'une image est la forme la plus simple de réhaussement, elle consiste à 1) optimiser les valeurs des pixels d'une image afin de maximiser la dynamique disponibles à l'affichage, ou 2) à changer le format de stockage des valeurs (de 8 bits à 16 bits):

$$ \text{nouvelle valeur d'un pixel} = \frac{\text{valeur d'un pixel} - min_0}{max_0 - min_0}\times (max_1 - min_1)+min_1$$ {#eq-rehauss-lin}

Par cette opération, on passe de la dynamique de départ ($max_0 - min_0$) vers la dynamique cible ($max_1 - min_1$). Bien que cette opération semble triviale, il est important d'être conscient des trois contraintes suivantes:

1.  **Faire attention à la dynamique cible**, ainsi, pour sauvegarder une image en format 8 bit, on utilisera alors $max_1=255$ et $min_1=0$.

2\. **Préservation de la valeur de no data** : il faut faire attention à la valeur $min_1$ dans le cas d'une valeur présente pour *no_data*. Par exemple, si *no_data=0* alors il faut s'assurer que $min_1>0$.

3\. **Précision du calcul** : si possible réaliser la division ci-dessus en format *float*

#### Cas des histogrammes asymétriques

Dans certains cas, la distribution de valeurs est très asymétrique et présente une longue queue avec des valeurs extrêmes élevées (à droite ou à gauche de l'histogramme). Le cas des images SAR est particulièrement représentatif de ce type de données. En effet, celles-ci peuvent présenter une distribution de valeurs de type exponentiel. Il est alors préférable d'utiliser des [percentiles](https://fr.wikipedia.org/wiki/Centile) au préalable afin d'explorer la forme de l'histogramme et la distribution des valeurs:

In [ ]:
NO_DATA_FLOAT= -999.0
# on prend tous les pixels de la première bande
values = img_SAR[0].values.flatten().astype(float)
# on exclut les valeurs invalides
values = values[~np.isnan(values)]
# on exclut le no data
values = values[values!=NO_DATA_FLOAT]
# calcul des percentiles
percentiles_position= (0,0.1,1,2,50,98,99,99.9,100)
percentiles= dict(zip(percentiles_position, np.percentile(values, percentiles_position)))
print(percentiles)

On constate que la valeur médiane (`0.012`) est très faible, ce qui signifie que 50% des valeurs sont inférieures à cette valeur alors que la valeur maximale (`483`) est 10 000 fois plus élevée! Une manière de visualiser cette distribution de valeurs est d'utiliser [`boxplot`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.boxplot.html) et [`violinplot`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.violinplot.html) de la librairie `matplotlib`:

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(6, 4), sharex=True)
ax[0].set_title('Distribution de la bande 0 de img_SAR', fontsize='small')
ax[0].grid(True)
ax[0].violinplot(values, orientation  ='horizontal', 
                 quantiles =(0.01,0.02,0.50,0.98,0.99),
                  showmeans=False,
                  showmedians=True)
ax[1].set_xlabel('Valeur des pixels')
ax[1].grid(True)
bplot = ax[1].boxplot(values, notch = True, orientation  ='horizontal')
plt.tight_layout()
plt.show()

Afin de visualiser correctement l'histogramme, il faut se limiter à un intervalle de valeurs plus réduit. Dans le code ci-dessous, on impose à la fonction `np.histogramme` de compter les valeurs de pixels dans des intervalles de valeurs fixés par la fonction `np.linspace(percentiles[0.1],percentiles[99.9], 50)` où `percentiles[0.1]` et `percentiles[99.9]` sont les $0.1\%$ et $99.9\%$ percentiles respectivement:

In [ ]:
hist, bin_edges = np.histogram(values, 
                                bins=np.linspace(percentiles[0.1], 
                                percentiles[99.9], 50), 
                                density=True)

fig, ax = plt.subplots(nrows=2,ncols=1,figsize=(6, 5), sharex=True)
ax[0].bar(bin_edges[:-1], 
                hist*(bin_edges[1]-bin_edges[0]), 
                width= (bin_edges[1]-bin_edges[0]), 
                edgecolor= 'w')
ax[0].set_title("Distribution de probabilité (PDF)")
ax[0].set_ylabel("Densité de probabilité")
ax[0].grid(True)

ax[1].plot(bin_edges[:-1], 
            hist.cumsum()*(bin_edges[1]-bin_edges[0]))
ax[1].set_title("Distribution de probabilité cumulée (CDF)")
ax[1].set_xlabel("Valeur du pixel")
ax[1].set_ylabel("Probabilité cumulée")
ax[1].grid(True)
plt.tight_layout()
plt.show()                              

Au niveau de l'affichage avec `matplotlib`, la dynamique peut être contrôlée directement avec les paramètres `vmin` et `vmax` comme ceci:

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(6, 5), sharex=True, sharey=True)
[a.axis('off') for a in ax.flatten()]
ax[0,0].imshow(img_SAR[0].values, vmin=percentiles[0], vmax=percentiles[100])
ax[0,0].set_title(f"0% - 100%={percentiles[0]:2.1f} - {percentiles[100]:2.1f}")
ax[0,1].imshow(img_SAR[0].values, vmin=percentiles[0.1], vmax=percentiles[99.9]) 
ax[0,1].set_title(f"0.1% - 99.9%={percentiles[0.1]:2.1f} - {percentiles[99.9]:2.1f}")
ax[1,0].imshow(img_SAR[0].values, vmin=percentiles[1], vmax=percentiles[99]) 
ax[1,0].set_title(f"1% - 99%={percentiles[1]:2.1f} - {percentiles[99]:2.1f}")
ax[1,1].imshow(img_SAR[0].values, vmin=percentiles[2], vmax=percentiles[98]) 
ax[1,1].set_title(f"2% - 98%={percentiles[2]:2.1f} - {percentiles[98]:2.1f}")
plt.tight_layout()

### Réhaussements non linéaires

#### Réhaussement par fonctions



Le réhaussenent par fonction consiste à appliquer une fonction non linéaire afin de modifier la dynamique de l'image. Par exemple, pour une image radar, une transformation populaire est d'afficher les valeurs de rétrodiffusion en décibel (`dB`) avec la fonction `log10()`.

In [ ]:
percentiles_position= (0,0.1,1,2,50,98,99,99.9,100)
sar= img_SAR[0].data
# on masque le no_data et les valeurs <= 0 (impropres au log) : sinon 10*log10 -> -inf/nan
valid= (sar != NO_DATA_FLOAT) & (sar > 0)
values= 10*np.log10(np.where(valid, sar, np.nan))  # image en dB, nan hors du masque
percentiles_db= dict(zip(percentiles_position, np.nanpercentile(values, percentiles_position)))
print(percentiles_db)

Les boites à moustache (*boxplots*) ont une bien meilleure distribution qui est en effet très proche d'une distribution normale gaussienne:

In [ ]:
values_valid= values.flatten()
values_valid= values_valid[np.isfinite(values_valid)]  # on retire les nan hors masque
fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(6, 4), sharex=True)
ax[0].set_title('Distribution de la bande 0 de img_SAR en dB', fontsize='small')
ax[0].grid(True)
ax[0].violinplot(values_valid, orientation  ='horizontal',
                 quantiles =(0.01,0.02,0.50,0.98,0.99),
                  showmeans=False,
                  showmedians=True,
                 showextrema = True)
ax[1].set_xlabel('Valeur des pixels')
ax[1].grid(True)
bplot = ax[1].boxplot(values_valid, notch = True, orientation  ='horizontal')
plt.tight_layout()
plt.show()

On obtient ainsi les images suivantes:

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(6, 5), sharex=True, sharey=True)
[a.axis('off') for a in ax.flatten()]
ax[0,0].imshow(values, vmin=percentiles_db[0], vmax=percentiles_db[100])
ax[0,0].set_title(f"0% - 100%={percentiles_db[0]:2.1f} - {percentiles_db[100]:2.1f}")
ax[0,1].imshow(values, vmin=percentiles_db[0.1], vmax=percentiles_db[99.9]) 
ax[0,1].set_title(f"0.1% - 99.9%={percentiles_db[0.1]:2.1f} - {percentiles_db[99.9]:2.1f}")
ax[1,0].imshow(values, vmin=percentiles_db[1], vmax=percentiles_db[99]) 
ax[1,0].set_title(f"1% - 99%={percentiles_db[1]:2.1f} - {percentiles_db[99]:2.1f}")
ax[1,1].imshow(values, vmin=percentiles_db[2], vmax=percentiles_db[98]) 
ax[1,1].set_title(f"2% - 98%={percentiles_db[2]:2.1f} - {percentiles_db[98]:2.1f}")
plt.tight_layout()

#### Réhaussement gamma (loi de puissance)

Une autre famille de réhaussements non linéaires très utilisée est la correction gamma (ou loi de puissance), qui applique un exposant $\gamma$ aux valeurs normalisées de l'image [@Jensen2016; @richards2022remote]:

$$ j = \left(\frac{i}{i_{max}}\right)^{\gamma} \times j_{max} $$ {#eq-rehauss-gamma}

Un $\gamma < 1$ éclaircit les tons foncés (utile pour une image sous-exposée) alors qu'un $\gamma > 1$ assombrit les tons clairs (utile pour une image surexposée); $\gamma = 1$ correspond à l'identité. Contrairement à l'étirement linéaire, cette transformation n'est pas symétrique entre les ombres et les hautes lumières:

In [ ]:
gammas = (0.5, 1.0, 2.0)
img_norm = np.clip(img_rgb.data.transpose(1, 2, 0) / 255.0, 0, 1)

fig, ax = plt.subplots(ncols=3, figsize=(9, 3))
for a, g in zip(ax, gammas):
    a.imshow(img_norm ** g)
    a.set_title(f"$\\gamma$={g}")
    a.axis('off')
plt.tight_layout()
plt.show()

#### Égalisation d'histogramme

L'égalisation d'histogramme consiste à modifier les valeurs des pixels d'une image source afin que la distribution cumulée des valeurs (CDF) devienne similaire à celle d'une image cible. La CDF (*Cumulative Distribution Function*) est simplement la somme cumulée des valeurs de l'histogramme:

$$
CDF_{source}(i)= \frac{1}{K}\sum_{j=0}^{j \leq i} hist_{source}(j)
$$ avec $K$ choisit de façon à ce que la dernière valeur soit égale à 1 ($CDF_{source}(i_{max})=1$). De la même manière, $CDF_{cible}$ est la CDF d'une image cible. La formule générale pour l'égalisation d'histogramme est la suivante: $$
j = CDF_{cible}^{-1}(CDF_{source}(i))
$$

On peut choisir $CDF_{cible}$ comme correspondant à une image où chaque valeur de pixel est équiprobable (d'où le terme *égalisation*), ce qui veut dire $hist_{cible}(j)=1/L$ avec $L$ égale au nombre de valeurs possibles dans l'image (par exemple $L=256$). $$
j = L \times CDF_{source}(i)
$$ On peut appliquer cette procédure sur l'image SAR en dB de la façon suivante:

In [ ]:
sar= img_SAR[0].data
# masque : no_data et valeurs <= 0 exclues avant le passage en dB
valid= (sar != NO_DATA_FLOAT) & (sar > 0)
sar_db= 10*np.log10(np.where(valid, sar, np.nan)) # image en dB (nan hors masque)
values= np.sort(sar_db[valid].flatten()) # valeurs valides rangées par ordre croissant
cdf_x= np.linspace(values[0], values[-1], 1000) # 1000 valeurs réparties également sur l'intervalle des valeurs
cdf_source= np.interp(cdf_x, values, np.arange(len(values))/len(values)*255) # calcul de la CDF source
# on applique sur la MÊME échelle (dB) que celle utilisée pour construire la CDF
values_eq= np.interp(sar_db, cdf_x, cdf_source)
values_eq= np.where(valid, values_eq, 0).astype('uint8') # no_data remis à 0
plt.imshow(values_eq)
plt.axis('off')

#### Égalisation adaptative (CLAHE)

L'égalisation globale calcule une seule CDF pour toute l'image, ce qui peut mal fonctionner lorsque le contraste varie localement (zones d'ombre et de forte lumière dans une même scène). L'égalisation adaptative à contraste limité (*Contrast Limited Adaptive Histogram Equalization*, CLAHE) découpe l'image en tuiles et égalise l'histogramme de chacune séparément, avec un plafond (`clip_limit`) qui évite d'amplifier le bruit dans les zones homogènes [@Jensen2016]. La librairie `scikit-image` en fournit une implémentation directe:

In [ ]:
from skimage import exposure

gray = img_rgb.data.transpose(1, 2, 0).mean(axis=2) / 255.0
img_global = exposure.equalize_hist(gray)
img_clahe = exposure.equalize_adapthist(gray, clip_limit=0.03)

fig, ax = plt.subplots(ncols=3, figsize=(9, 3))
for a, im, title in zip(ax, (gray, img_global, img_clahe),
                         ('originale', 'égalisation globale', 'CLAHE')):
    a.imshow(im, vmin=0, vmax=1)
    a.set_title(title)
    a.axis('off')
plt.tight_layout()
plt.show()

La CLAHE fait ressortir davantage de détails locaux (textures, zones d'ombre) sans saturer les zones déjà bien contrastées, contrairement à l'égalisation globale.

#### Correspondance d'histogrammes

L'égalisation d'histogramme est en fait un cas particulier d'un problème plus général: faire correspondre la CDF d'une image source à une CDF cible arbitraire, et non uniquement à une distribution uniforme. Cette technique, la correspondance d'histogrammes (*histogram matching*), est notamment utile pour harmoniser la dynamique entre deux acquisitions, par exemple deux scènes adjacentes à mosaïquer [@richards2022remote; @Schowengerdt2007]. La fonction `match_histograms` de `scikit-image` implémente directement $j = CDF_{cible}^{-1}(CDF_{source}(i))$ pour une CDF cible quelconque, ici une distribution gaussienne:

In [ ]:
from skimage.exposure import match_histograms

source = np.log10(img_SAR[0].data)
cible = np.random.normal(loc=source.mean(), scale=source.std(), size=source.shape)
source_matched = match_histograms(source, cible)

fig, ax = plt.subplots(ncols=2, figsize=(7, 3.5))
ax[0].imshow(source)
ax[0].set_title("originale (dB)")
ax[1].imshow(source_matched)
ax[1].set_title("après correspondance (cible gaussienne)")
[a.axis('off') for a in ax]
plt.tight_layout()
plt.show()

#### Palettes de couleur

Les palettes de couleurs sont appliquées dynamiquement à l'affichage sur une image à une seule bande. La librairie `matplotlib` contient un nombre considérable de [palettes](https://matplotlib.org/stable/users/explain/colors/colormaps.html).

In [ ]:
# | output: false
from matplotlib import colormaps
list(colormaps)

Voici quelques exemples ci-dessous, les valeurs de l'image doivent être normalisées entre 0 et 1 ou entre 0 et 255 sinon les paramètres `vmin` et `vmax` doivent être spécifiés. On peut observer comment ces palettes révèlent les détails de l'image malgré une image originalement très sombre.

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(6, 5), sharex=True, sharey=True)
[a.axis('off') for a in ax.flatten()]
ax[0,0].imshow(img_SAR[0].data, vmin=percentiles[2], vmax=percentiles[98], cmap='jet')
ax[0,0].set_title(f"jet")
ax[0,1].imshow(img_SAR[0].data, vmin=percentiles[2], vmax=percentiles[98], cmap='hot')
ax[0,1].set_title(f"hot")
ax[1,0].imshow(img_SAR[0].data, vmin=percentiles[2], vmax=percentiles[98], cmap='hsv')
ax[1,0].set_title(f"hsv")
ax[1,1].imshow(img_SAR[0].data, vmin=percentiles[2], vmax=percentiles[98], cmap='terrain')
ax[1,1].set_title(f"terrain")
plt.tight_layout()

Il peut être utile d'ajouter une barre de couleurs afin d'indiquer la correspondance entre les couleurs et les valeurs numériques:

In [ ]:
import matplotlib as mpl
fig, ax = plt.subplots(figsize=(6, 6))
cmap= mpl.colormaps.get_cmap('jet').with_extremes(under='white', over='magenta')
h=plt.imshow(img_SAR[0].data, norm=mpl.colors.LogNorm(vmin=percentiles[2], vmax=percentiles[98]),
                   cmap=cmap)
fig.colorbar(h, ax=ax,  orientation='horizontal', label="Intensité", extend='both')
ax.axis('off') 

### Composés colorés

Le système visuel humain est sensible seulement à la partie visible du spectre électromagnétique qui compose les couleurs de l'arc-en-ciel du bleu au rouge. L'ensemble des couleurs du spectre visible peut être obtenu à partir du mélange de trois couleurs primaires (rouge, vert et bleu). Ce système de décomposition à trois couleurs est à la base de la plupart des systèmes de visualisation ou de représentation de l'information de couleur. Si on prend le cas des images Sentinel-2, 12 bandes sont disponibles, plusieurs composés couleurs sont donc possibles (voir le site de [Copernicus](https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/composites/)). Voici quelques exemples possibles, chaque composé mettant en valeur des propriétés différentes de la surface.

In [ ]:
import rioxarray as rxr
fig, ax= plt.subplots(nrows=2, ncols= 2, figsize=(8, 6), sharex=True, sharey=True)
img_s2.sel(band=[4,3,2]).plot.imshow(vmin=86, vmax=4000, ax=ax[0,0])
ax[0,0].set_title('RVB')
img_s2.sel(band=[8,3,2]).plot.imshow(vmin=86, vmax=4000, ax=ax[0,1])
ax[0,1].set_title('NIR,V,B')
img_s2.sel(band=[12,8,4]).plot.imshow(vmin=86, vmax=4000, ax=ax[1,0])
ax[1,0].set_title('SWIR2,NIR,R')
img_s2.sel(band=[12,11,4]).plot.imshow(vmin=86, vmax=4000, ax=ax[1,1])
ax[1,1].set_title('SWIR2,SWIR1,NIR')
plt.tight_layout()
plt.show()

#### Étirement par décorrélation

Les bandes d'une image multispectrale sont souvent fortement corrélées entre elles, ce qui donne des composés couleurs peu contrastés (dominante grisâtre). L'étirement par décorrélation (*decorrelation stretch*) corrige ce problème en décorrélant les bandes dans l'espace des composantes principales, en égalisant leur variance, puis en revenant dans l'espace original [@Schowengerdt2007; @richards2022remote]:

In [ ]:
def decorrelation_stretch(img):
    # img: (bandes, y, x)
    X = img.reshape(img.shape[0], -1).astype(float)
    X -= X.mean(axis=1, keepdims=True)
    valeurs, vecteurs = np.linalg.eigh(np.cov(X))
    X_pca = (vecteurs.T @ X) / np.sqrt(valeurs)[:, None]
    X_stretch = vecteurs @ X_pca
    q= np.quantile(X_stretch, [0.01, 0.02, 0.98, 0.99])
    print(q)
    X_stretch -= q[1]
    X_stretch /= (q[2]-q[1])
    return X_stretch.clip(0,1).reshape(img.shape)

composite = img_s2.sel(band=[4, 3, 2]).data
composite_stretch = decorrelation_stretch(composite)

fig, ax = plt.subplots(ncols=2, figsize=(8, 4))
ax[0].imshow(np.clip(composite.transpose(1, 2, 0) / 4000.0, 0, 1))
ax[0].set_title("composé RVB original")
ax[1].imshow(composite_stretch.transpose(1, 2, 0))
ax[1].set_title("après étirement par décorrélation")
[a.axis('off') for a in ax]
plt.tight_layout()
plt.show()

Le résultat conserve les teintes relatives entre bandes tout en maximisant le contraste de chacune des composantes principales, ce qui fait ressortir davantage de détails que le composé original.

On peut confirmer cet étalement en traçant l'histogramme des trois bandes du composé étiré avec `show_hist` de `rasterio`, qui accepte directement une matrice `(bandes, lignes, colonnes)`. Après décorrélation, chaque bande occupe désormais toute la plage `[0, 1]` :

In [ ]:
from rasterio.plot import show_hist

fig, ax = plt.subplots(figsize=(6, 4))
show_hist(composite_stretch, bins=50, lw=0.0, stacked=False, alpha=0.3,
          histtype='stepfilled', ax=ax,
          title="Histogramme du composé après étirement par décorrélation")
plt.show()

##### Poids des bandes : le cercle des corrélations

Au-delà du résultat visuel, l'ACP nous renseigne sur **la façon dont chaque bande contribue à chaque composante principale**. Les vecteurs propres donnent le **poids** de chaque bande dans une composante ; multipliés par la racine carrée de la valeur propre associée, ils fournissent la **corrélation** entre chaque bande et chaque composante. On visualise ces corrélations dans un **cercle des corrélations** : chaque bande devient une flèche partant de l'origine, dont les coordonnées sont ses corrélations avec les deux premières composantes (CP1 et CP2). Une flèche proche du cercle unité est bien représentée dans le plan CP1-CP2 ; deux flèches proches signalent des bandes corrélées, deux flèches opposées des bandes anti-corrélées.

In [ ]:
bandes = [2, 3, 4, 8, 11, 12]
noms = ['B', 'V', 'R', 'PIR', 'SWIR1', 'SWIR2']
X = img_s2.sel(band=bandes).data.reshape(len(bandes), -1).astype(float)
X = X[:, ~np.isnan(X).any(axis=0)]                              # on retire les no_data
X = (X - X.mean(1, keepdims=True)) / X.std(1, keepdims=True)    # standardisation

valeurs, vecteurs = np.linalg.eigh(np.corrcoef(X))             # ACP sur la corrélation
ordre = np.argsort(valeurs)[::-1]                               # variance décroissante
valeurs, vecteurs = valeurs[ordre], vecteurs[:, ordre]
loadings = vecteurs * np.sqrt(valeurs)                         # corrélation bande <-> composante
pct = 100 * valeurs / valeurs.sum()

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.add_patch(plt.Circle((0, 0), 1, fill=False, color='grey', ls='--'))
ax.axhline(0, color='grey', lw=0.5); ax.axvline(0, color='grey', lw=0.5)
for i, nom in enumerate(noms):
    ax.arrow(0, 0, loadings[i, 0], loadings[i, 1], color='tab:blue',
             head_width=0.03, length_includes_head=True)
    ax.text(loadings[i, 0] * 1.15, loadings[i, 1] * 1.15, nom, ha='center', va='center')
ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2); ax.set_aspect('equal')
ax.set_xlabel(f"CP1 ({pct[0]:.0f} %)"); ax.set_ylabel(f"CP2 ({pct[1]:.0f} %)")
ax.set_title("Cercle des corrélations (ACP sur 6 bandes Sentinel-2)")
plt.tight_layout()
plt.show()

Sur ce composé Sentinel-2, la première composante (CP1, environ 68 % de la variance) est corrélée négativement à presque toutes les bandes : elle traduit la **luminosité globale** de la scène. La seconde (CP2, environ 27 %) oppose le **proche infrarouge et le SWIR1** au reste : c'est un axe de **végétation / humidité**. Les bandes visibles (B, V, R), dont les flèches sont quasi confondues, sont **fortement corrélées** entre elles — c'est précisément cette redondance que l'étirement par décorrélation vient corriger.

## Points clés


## Exercices


**À vous de jouer**

1.  Proposez une autre transformation non linéaire pour l'image SAR (p. ex. la racine carrée ou `np.arcsinh`) et comparez son histogramme à celui obtenu en décibels.

2.  À l'aide de `skimage.exposure.match_histograms`, faites correspondre l'histogramme de la bande proche infrarouge de `RGBNIR_of_S2A.tif` à celui d'une bande de `sentinel2.tif`. Discutez du résultat.

3.  À partir de `img_s2`, construisez un nouveau composé coloré (p. ex. `[11, 8, 4]` ou `[8, 4, 3]`) et décrivez les surfaces qu'il met en valeur.

4.  *(visualisation web)* Dans Colab, installez `leafmap`, chargez `RGBNIR_of_S2A.tif` en composé infrarouge (`indexes=[4, 3, 2]`) sur un fond `Esri.WorldImagery`, puis comparez vraie couleur et infrarouge avec `split_map`.

5.  Comparez une égalisation d'histogramme globale et une égalisation adaptative (CLAHE) sur une image de votre choix. Dans quels cas la CLAHE fait-elle ressortir des détails invisibles avec l'égalisation globale?

6.  Appliquez un étirement par décorrélation sur le composé SWIR2, NIR, R de `sentinel2.tif` et comparez-le au composé sans étirement. La corrélation entre bandes est-elle plus ou moins forte que pour le composé RVB naturel?

7.  *(cercle des corrélations)* Reprenez le **cercle des corrélations** sur les **quatre bandes** de `RGBNIR_of_S2A.tif` (B, V, R, PIR) : standardisez les bandes, calculez l'ACP sur leur matrice de corrélation, puis tracez les flèches des *loadings* dans le plan CP1-CP2. Quelle(s) bande(s) domine(nt) la première composante? La flèche du proche infrarouge est-elle alignée avec celles du visible ou s'en écarte-t-elle nettement? Qu'en concluez-vous sur la corrélation entre le PIR et les bandes visibles?

## Quiz


::: {.content-visible when-profile="production"}

Utilisez la version html.
:::


In [ ]:
from code_complementaire.quizz_functions import Quiz, render_quizz
Chap02Quiz = Quiz("quiz/Chap02.yml", "Chap02")
render_quizz(Chap02Quiz)